le nom: El jattioui
le prenom: Maryame
Master:GLCC

In [1]:
import numpy as np
from collections import Counter

# =================================================================
# 1. FONCTIONS MATHÉMATIQUES DE BASE
# =================================================================

def entropy(y):
    """
    Calcule l'Entropie. 
    L'entropie mesure le désordre : 
    - Si toutes les données sont de la même classe -> Entropie = 0
    - Si les données sont mélangées à 50/50 -> Entropie = 1
    """
    # On compte le nombre d'occurrences de chaque classe (0 ou 1)
    hist = np.bincount(y)
    # On transforme ces comptes en probabilités (fréquences)
    ps = hist / len(y)
    # Formule : - Somme de (p * log2(p))
    return -np.sum([p * np.log2(p) for p in ps if p > 0])

class Node:
    """
    Représente un élément de l'arbre.
    Un nœud est soit une QUESTION (nœud interne) soit une RÉPONSE (feuille).
    """
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature     # L'indice de la colonne testée (ex: Humidité)
        self.threshold = threshold # La valeur de coupure (ex: > 80)
        self.left = left           # Branche si le test est VRAI
        self.right = right         # Branche si le test est FAUX
        self.value = value         # Stocke la classe finale (uniquement pour les feuilles)

# =================================================================
# 2. CONSTRUCTION DE L'ARBRE (Logique Récursive)
# =================================================================
class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth # Profondeur maximale pour éviter l'overfitting
        self.min_samples_split = min_samples_split # Nb minimum de données pour diviser
        self.root = None # Racine de l'arbre

    def fit(self, X, y):
        #Lance la construction de l'arbre
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        # Méthode récursive pour créer les branches
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # --- CONDITIONS D'ARRÊT ---
        # On s'arrête si : l'arbre est trop profond, ou s'il n'y a qu'une seule classe
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)

        # --- RECHERCHE DU MEILLEUR SPLIT ---
        # On teste chaque caractéristique pour trouver celle qui réduit le mieux l'entropie
        feat_idxs = np.random.choice(n_features, n_features, replace=False)
        
        best_feat, best_thresh = self._best_criteria(X, y, feat_idxs)

        # --- CRÉATION DES BRANCHES ---
        # On divise les données en deux groupes selon le meilleur seuil trouvé
        left_idxs, right_idxs = self. _split(X[:, best_feat], best_thresh)
        
        # On recommence le processus pour la branche gauche et droite (Récursivité)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)

    def _best_criteria(self, X, y, feat_idxs):
        """Cherche la paire (caractéristique, seuil) avec le meilleur Gain d'Information."""
        best_gain = -1
        split_idx, split_thresh = None, None
        
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            for threshold in thresholds:
                # On calcule le gain pour chaque seuil possible   
                gain = self._information_gain(y, X_column, threshold)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = threshold
        return split_idx, split_thresh

    def _information_gain(self, y, X_column, split_thresh):
        """
        Le Gain d'Information mesure la réduction de l'entropie.
        Gain = Entropie(Parent) - [Somme pondérée Entropie(Enfants)]
        """
        parent_entropy = entropy(y)
        
        # On crée les groupes temporaires pour calculer leur entropie
        left_idxs, right_idxs = self._split(X_column, split_thresh)
        if len(left_idxs) == 0 or len(right_idxs) == 0: return 0
        
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = entropy(y[left_idxs]), entropy(y[right_idxs])
        
        # Moyenne pondérée de l'entropie des deux nouveaux groupes
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r
        return parent_entropy - child_entropy

    def _split(self, X_column, split_thresh):
        """Sépare les indices des données selon le seuil."""
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def predict(self, X):
        """Prédit la classe pour un ensemble de données."""
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        """Parcourt l'arbre de la racine vers les feuilles pour un point donné."""
        # Si on est sur une feuille, on renvoie sa valeur
        if node.value is not None: 
            return node.value


        
        # Sinon, on suit la branche gauche ou droite selon le test
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

# =================================================================
# 3. EXEMPLE D'UTILISATION
# =================================================================
# Données : [Température, Humidité] -> Jouer au Tennis (1: Oui, 0: Non)
X_train = np.array([[25, 80], [22, 90], [30, 70], [15, 60], [18, 95], [20, 70]])
y_train = np.array([1, 0, 1, 1, 0, 1])

# Initialisation et entraînement
clf = DecisionTree(max_depth=3)
clf.fit(X_train, y_train)

# Test sur une nouvelle journée (21 degrés, 85% d'humidité)
prediction = clf.predict([[21, 85]])
print(f"Prédiction pour [21°C, 85% Humidité] : {'Jouer' if prediction[0]==1 else 'Ne pas jouer'}")

Prédiction pour [21°C, 85% Humidité] : Ne pas jouer
